In [2]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

# 设置参数
SNR_DB = 20  # 目标信噪比(dB)，可根据需要调整
SEED = 42  # 随机种子确保可重复性
np.random.seed(SEED)

# 1. 读取原始数据
print("正在读取原始数据...")
with h5py.File('CWRU_base.h5', 'r') as f:
    # 读取原始数据
    X_train = f['X_train'][:]
    X_val = f['X_val'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_val = f['y_val'][:]
    y_test = f['y_test'][:]
    
print(f"数据读取完成: 训练集 {X_train.shape}, 验证集 {X_val.shape}, 测试集 {X_test.shape}")

# 2. 定义自适应噪声添加函数
def add_adaptive_noise(data, snr_db=SNR_DB):
    """
    添加自适应噪声，根据信噪比(SNR)动态调整噪声强度
    snr_db: 目标信噪比(dB)
    """
    # 创建数据副本以避免修改原始数据
    noisy_data = np.copy(data)
    
    # 获取数据形状信息
    n_samples = data.shape[0]
    signal_length = data.shape[2]
    
    # 添加进度条
    for i in tqdm(range(n_samples), desc="添加自适应噪声"):
        # 提取当前样本信号 (形状: [1, 1024, 1])
        signal = data[i, 0, :, 0]
        
        # 计算信号功率 (均方值)
        signal_power = np.mean(signal ** 2)
        
        # 跳过零功率信号
        if signal_power < 1e-10:
            noisy_data[i] = data[i]
            continue
            
        # 计算目标噪声功率
        snr_linear = 10 ** (snr_db / 10)
        noise_power = signal_power / snr_linear
        
        # 生成高斯噪声
        noise = np.random.normal(0, np.sqrt(noise_power), signal_length)
        
        # 添加噪声到信号
        noisy_signal = signal + noise
        
        # 更新数据
        noisy_data[i, 0, :, 0] = noisy_signal
    
    return noisy_data

# 3. 计算信噪比的函数
def calculate_snr(original, noisy):
    """
    计算实际信噪比(SNR)
    original: 原始信号
    noisy: 添加噪声后的信号
    """
    # 确保信号形状一致
    original = original.flatten()
    noisy = noisy.flatten()
    
    # 计算噪声分量
    noise = noisy - original
    
    # 计算信号功率和噪声功率
    signal_power = np.mean(original ** 2)
    noise_power = np.mean(noise ** 2)
    
    # 避免除以零
    if noise_power < 1e-10:
        return float('inf')
    
    # 计算SNR(dB)
    snr_db = 10 * np.log10(signal_power / noise_power)
    return snr_db

def calculate_snr_for_dataset(original_data, noisy_data):
    """
    计算整个数据集的平均信噪比
    """
    snr_values = []
    
    for i in range(original_data.shape[0]):
        original_signal = original_data[i, 0, :, 0]
        noisy_signal = noisy_data[i, 0, :, 0]
        snr = calculate_snr(original_signal, noisy_signal)
        if snr != float('inf'):
            snr_values.append(snr)
    
    if snr_values:
        return np.mean(snr_values), np.std(snr_values), snr_values
    else:
        return float('inf'), 0, []

# 4. 只在测试集添加噪声
print("\n只在测试集添加噪声...")
X_test_noisy = add_adaptive_noise(X_test, snr_db=SNR_DB)

# 5. 计算测试集的实际信噪比
print("\n计算测试集的实际信噪比...")
test_snr_mean, test_snr_std, test_snr_values = calculate_snr_for_dataset(X_test, X_test_noisy)
print(f"测试集实际平均信噪比: {test_snr_mean:.2f} ± {test_snr_std:.2f} dB")

# 6. 可视化信噪比分布
def plot_snr_distribution(snr_values, title, filename):
    """绘制信噪比分布图"""
    plt.figure(figsize=(10, 6))
    
    # 绘制直方图
    plt.hist(snr_values, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    
    # 添加平均线和标注
    mean_snr = np.mean(snr_values)
    plt.axvline(mean_snr, color='red', linestyle='dashed', linewidth=2, 
                label=f'平均值: {mean_snr:.2f} dB')
    
    # 添加目标SNR线
    plt.axvline(SNR_DB, color='green', linestyle='dashed', linewidth=2, 
                label=f'目标SNR: {SNR_DB} dB')
    
    plt.xlabel('信噪比 (dB)')
    plt.ylabel('样本数量')
    plt.title(f'{title} - 信噪比分布')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

# 绘制测试集的信噪比分布
plot_snr_distribution(test_snr_values, '测试集', 'test_snr_distribution.png')

# 7. 保存数据集
def save_h5_dataset(filename, X_train, X_val, X_test, y_train, y_val, y_test):
    """保存数据集到HDF5文件"""
    with h5py.File(filename, 'w') as f:
        f.create_dataset('X_train', data=X_train)
        f.create_dataset('X_val', data=X_val)
        f.create_dataset('X_test', data=X_test)
        f.create_dataset('y_train', data=y_train)
        f.create_dataset('y_val', data=y_val)
        f.create_dataset('y_test', data=y_test)
    print(f"已保存: {filename}")

print("\n保存数据集...")
# 只在测试集加噪声
save_h5_dataset('CWRU_test_noisy_SNR20.h5', 
                X_train, X_val, X_test_noisy,
                y_train, y_val, y_test)

# 8. 可视化函数 - 用于单个样本对比
def plot_single_sample_comparison(original, noisy, title, filename):
    """绘制单个样本的原始信号和噪声信号对比"""
    # 计算该样本的实际SNR
    snr = calculate_snr(original, noisy)
    
    plt.figure(figsize=(12, 8))
    
    # 原始信号
    plt.subplot(3, 1, 1)
    plt.plot(original)
    plt.title(f"Original Signal - {title}")  # 原始信号
    plt.xlabel("Time Points")  # 时间点
    plt.ylabel("Amplitude")  # 振幅
    
    # 噪声信号
    plt.subplot(3, 1, 2)
    plt.plot(noisy)
    plt.title(f"Noisy Signal (实际SNR={snr:.2f}dB, 目标SNR={SNR_DB}dB) - {title}")  # 添加噪声后的信号
    plt.xlabel("Time Points")  # 时间点
    plt.ylabel("Amplitude")  # 振幅
    
    # 噪声分量
    plt.subplot(3, 1, 3)
    noise_component = noisy - original
    plt.plot(noise_component)
    plt.title(f"Noise Component - {title}")  # 噪声分量
    plt.xlabel("Time Points")  # 时间点
    plt.ylabel("Amplitude")  # 振幅
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()  # 关闭图形以节省内存

# 9. 对测试集进行可视化对比
print("\n开始对测试集进行可视化对比...")
sample_idx = np.random.randint(0, 2200)  # 随机选择一个样本

# 测试集样本对比
plot_single_sample_comparison(
    X_test[sample_idx, 0, :, 0].flatten(),
    X_test_noisy[sample_idx, 0, :, 0].flatten(),
    "测试集样本",
    'signal_comparison_test.png'
)

# 10. 创建综合对比图
print("\n创建综合对比图...")
def create_comprehensive_comparison():
    """创建测试集信号的综合对比图"""
    plt.figure(figsize=(15, 10))
    
    # 原始测试集样本
    plt.subplot(2, 3, 1)
    plt.plot(X_test[sample_idx, 0, :, 0].flatten())
    plt.title("原始测试集样本")
    plt.xlabel("时间点")
    plt.ylabel("振幅")
    
    # 噪声测试集样本
    plt.subplot(2, 3, 2)
    plt.plot(X_test_noisy[sample_idx, 0, :, 0].flatten())
    plt.title("噪声测试集样本")
    plt.xlabel("时间点")
    
    # 噪声分量
    plt.subplot(2, 3, 3)
    noise = X_test_noisy[sample_idx, 0, :, 0].flatten() - X_test[sample_idx, 0, :, 0].flatten()
    plt.plot(noise)
    plt.title("噪声分量")
    plt.xlabel("时间点")
    
    # 信噪比分布
    plt.subplot(2, 3, 4)
    plt.hist(test_snr_values, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    mean_snr = np.mean(test_snr_values)
    plt.axvline(mean_snr, color='red', linestyle='dashed', linewidth=2, 
                label=f'平均值: {mean_snr:.2f} dB')
    plt.axvline(SNR_DB, color='green', linestyle='dashed', linewidth=2, 
                label=f'目标SNR: {SNR_DB} dB')
    plt.xlabel('信噪比 (dB)')
    plt.ylabel('样本数量')
    plt.title('测试集信噪比分布')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 信号功率谱密度对比
    plt.subplot(2, 3, 5)
    f_orig, Pxx_orig = plt.psd(X_test[sample_idx, 0, :, 0].flatten(), Fs=12000, visible=False)
    f_noisy, Pxx_noisy = plt.psd(X_test_noisy[sample_idx, 0, :, 0].flatten(), Fs=12000, visible=False)
    plt.cla()  # 清除之前的绘图
    plt.semilogy(f_orig, Pxx_orig, label='原始信号')
    plt.semilogy(f_noisy, Pxx_noisy, label='噪声信号')
    plt.xlabel('频率 (Hz)')
    plt.ylabel('PSD (V²/Hz)')
    plt.title('功率谱密度对比')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 信号统计信息
    plt.subplot(2, 3, 6)
    plt.axis('off')  # 关闭坐标轴
    stats_text = f"""
    信号统计信息:
    - 目标SNR: {SNR_DB} dB
    - 实际平均SNR: {test_snr_mean:.2f} dB
    - SNR标准差: {test_snr_std:.2f} dB
    - SNR范围: {np.min(test_snr_values):.2f} - {np.max(test_snr_values):.2f} dB
    - 样本总数: {len(test_snr_values)}
    """
    plt.text(0.1, 0.5, stats_text, fontsize=10, verticalalignment='center')
    plt.title('统计信息')
    
    plt.tight_layout()
    plt.savefig('comprehensive_test_signal_comparison.png', dpi=300)
    plt.close()

create_comprehensive_comparison()

# 11. 验证文件内容
def verify_h5_file(filename):
    """验证HDF5文件内容"""
    print(f"\n验证文件: {filename}")
    with h5py.File(filename, 'r') as f:
        print(f"X_train shape: {f['X_train'].shape}")
        print(f"X_test shape: {f['X_test'].shape}")
        print(f"y_train shape: {f['y_train'].shape}")
        print(f"y_test shape: {f['y_test'].shape}")
        
        # 计算信噪比
        sample_idx = np.random.randint(0, 2200)
        orig_signal = f['X_test'][sample_idx, 0, :, 0]
        
        # 获取原始数据中的对应信号
        with h5py.File('CWRU_base.h5', 'r') as orig_f:
            orig_signal_clean = orig_f['X_test'][sample_idx, 0, :, 0]
        
        noise = orig_signal - orig_signal_clean
        signal_power = np.mean(orig_signal_clean ** 2)
        noise_power = np.mean(noise ** 2)
        
        if noise_power > 1e-10:
            snr = 10 * np.log10(signal_power / noise_power)
            print(f"测试集样本 {sample_idx} 实际信噪比: {snr:.2f} dB")
        else:
            print("噪声功率过低，无法计算SNR")

print("\n验证生成的文件...")
verify_h5_file('CWRU_test_noisy_SNR20.h5')

# 12. 打印信噪比统计
print(f"\n信噪比统计:")
print(f"测试集实际平均信噪比: {test_snr_mean:.2f} ± {test_snr_std:.2f} dB")
print(f"目标信噪比: {SNR_DB} dB")
print(f"测试集信噪比范围: {np.min(test_snr_values):.2f} - {np.max(test_snr_values):.2f} dB")

# 13. 保存信噪比统计数据到文件
with open('test_snr_statistics.txt', 'w') as f:
    f.write("测试集信噪比统计报告\n")
    f.write("=" * 50 + "\n")
    f.write(f"目标信噪比: {SNR_DB} dB\n")
    f.write(f"实际平均信噪比: {test_snr_mean:.2f} ± {test_snr_std:.2f} dB\n")
    f.write(f"信噪比范围: {np.min(test_snr_values):.2f} - {np.max(test_snr_values):.2f} dB\n")
    f.write(f"样本总数: {len(test_snr_values)}\n")
    f.write("\n")
    f.write("详细信噪比分布 (前20个样本):\n")
    for i, snr in enumerate(test_snr_values[:20]):
        f.write(f"  样本 {i}: {snr:.2f} dB\n")
    f.write("...\n")

print("\n所有操作完成！已保存测试集添加噪声的数据集和可视化对比图。")

正在读取原始数据...
数据读取完成: 训练集 (2200, 1, 1024, 1), 验证集 (2200, 1, 1024, 1), 测试集 (2200, 1, 1024, 1)

只在测试集添加噪声...


添加自适应噪声: 100%|███████████████████████████████████████████████████████████| 2200/2200 [00:00<00:00, 20823.77it/s]
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:142: UserWarning: Glyph 20449 (\N{CJK UNIFIED IDEOGRAPH-4FE1}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:142: UserWarning: Glyph 22122 (\N{CJK UNIFIED IDEOGRAPH-566A}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:142: UserWarning: Glyph 27604 (\N{CJK UNIFIED IDEOGRAPH-6BD4}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:142: UserWarning: Glyph 26679 (\N{CJK UNIFIED IDEOGRAPH-6837}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:142: UserWarning: Glyph 26412 (\N{CJK UNIFIED IDEOGRAPH-672C}) missing from current font.
  plt.tight_layout()
C:\Users\110


计算测试集的实际信噪比...
测试集实际平均信噪比: 20.00 ± 0.19 dB


C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:143: UserWarning: Glyph 26679 (\N{CJK UNIFIED IDEOGRAPH-6837}) missing from current font.
  plt.savefig(filename, dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:143: UserWarning: Glyph 26412 (\N{CJK UNIFIED IDEOGRAPH-672C}) missing from current font.
  plt.savefig(filename, dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:143: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from current font.
  plt.savefig(filename, dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:143: UserWarning: Glyph 37327 (\N{CJK UNIFIED IDEOGRAPH-91CF}) missing from current font.
  plt.savefig(filename, dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:143: UserWarning: Glyph 24179 (\N{CJK UNIFIED IDEOGRAPH-5E73}) missing from current font.
  plt.savefig(filename, dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:1


保存数据集...
已保存: CWRU_test_noisy_SNR20.h5

开始对测试集进行可视化对比...


C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:197: UserWarning: Glyph 27979 (\N{CJK UNIFIED IDEOGRAPH-6D4B}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:197: UserWarning: Glyph 35797 (\N{CJK UNIFIED IDEOGRAPH-8BD5}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:197: UserWarning: Glyph 38598 (\N{CJK UNIFIED IDEOGRAPH-96C6}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:197: UserWarning: Glyph 26679 (\N{CJK UNIFIED IDEOGRAPH-6837}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:197: UserWarning: Glyph 26412 (\N{CJK UNIFIED IDEOGRAPH-672C}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:197: UserWarning: Glyph 23454 (\N{CJK UNIFIED IDEOGRAPH-5B9E}


创建综合对比图...


C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:258: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  plt.semilogy(f_orig, Pxx_orig, label='原始信号')
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:280: UserWarning: Glyph 26102 (\N{CJK UNIFIED IDEOGRAPH-65F6}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:280: UserWarning: Glyph 38388 (\N{CJK UNIFIED IDEOGRAPH-95F4}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:280: UserWarning: Glyph 28857 (\N{CJK UNIFIED IDEOGRAPH-70B9}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:280: UserWarning: Glyph 25391 (\N{CJK UNIFIED IDEOGRAPH-632F}) missing from current font.
  plt.tight_layout()
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:280: UserWarning: Glyph 24133 (\N{CJK UNIF

C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:281: UserWarning: Glyph 21151 (\N{CJK UNIFIED IDEOGRAPH-529F}) missing from current font.
  plt.savefig('comprehensive_test_signal_comparison.png', dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:281: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from current font.
  plt.savefig('comprehensive_test_signal_comparison.png', dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:281: UserWarning: Glyph 35889 (\N{CJK UNIFIED IDEOGRAPH-8C31}) missing from current font.
  plt.savefig('comprehensive_test_signal_comparison.png', dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:281: UserWarning: Glyph 23494 (\N{CJK UNIFIED IDEOGRAPH-5BC6}) missing from current font.
  plt.savefig('comprehensive_test_signal_comparison.png', dpi=300)
C:\Users\11056\AppData\Local\Temp\ipykernel_79808\4098172141.py:281: UserWarning: Glyph 24230 (\N{CJK UNIFIED IDEOGR


验证生成的文件...

验证文件: CWRU_test_noisy_SNR20.h5
X_train shape: (2200, 1, 1024, 1)
X_test shape: (2200, 1, 1024, 1)
y_train shape: (2200,)
y_test shape: (2200,)
测试集样本 702 实际信噪比: 20.43 dB

信噪比统计:
测试集实际平均信噪比: 20.00 ± 0.19 dB
目标信噪比: 20 dB
测试集信噪比范围: 19.17 - 20.66 dB

所有操作完成！已保存测试集添加噪声的数据集和可视化对比图。
